# Cox Proportional Hazards Analysis: Captain to Major Promotion

This notebook analyzes time to promotion from Captain to Major for US Army officers using Cox proportional hazards model with time-dependent covariates.

## Objectives:
- Predict time from Captain promotion to Major promotion
- Account for time-dependent covariates: marriage status, job code changes
- Handle different types of censoring: study end, early departure
- Examine effects of sex, age, marriage timing, and MOS changes


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

# Survival analysis libraries
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Plotting settings
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)


In [2]:
# Load the data
df = pd.read_csv('cox_model/cox_data.csv')

# Basic data exploration
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head(10)


Dataset shape: (312, 23)

Column names:
['snpsht_dt', 'pid_pde', 'rank_pde', 'sex', 'age', 'job_code', 'married', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22']

First few rows:


,snpsht_dt,pid_pde,rank_pde,sex,age,job_code,married,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22
0,3/31/2001,PDE01,CPT,1.0,27.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6/30/2001,PDE01,CPT,1.0,28.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9/30/2001,PDE01,CPT,1.0,28.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12/31/2001,PDE01,CPT,1.0,28.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3/31/2002,PDE01,CPT,1.0,28.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6/30/2002,PDE01,CPT,1.0,29.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9/30/2002,PDE01,CPT,1.0,29.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,12/31/2002,PDE01,CPT,1.0,29.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,3/31/2003,PDE01,CPT,1.0,29.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,6/30/2003,PDE01,CPT,1.0,30.0,11A,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Clean the data - remove empty rows and extra columns
df_clean = df.dropna(subset=['snpsht_dt']).copy()
df_clean = df_clean[['snpsht_dt', 'pid_pde', 'rank_pde', 'sex', 'age', 'job_code', 'married']]

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Unique officers: {df_clean['pid_pde'].nunique()}")
print(f"Date range: {df_clean['snpsht_dt'].min()} to {df_clean['snpsht_dt'].max()}")

# Convert date column
df_clean['snpsht_dt'] = pd.to_datetime(df_clean['snpsht_dt'])

# Sort by officer and date
df_clean = df_clean.sort_values(['pid_pde', 'snpsht_dt']).reset_index(drop=True)

df_clean.head()


Cleaned dataset shape: (299, 7)
Unique officers: 13
Date range: 12/31/2001 to 9/30/2007


,snpsht_dt,pid_pde,rank_pde,sex,age,job_code,married
0,2001-03-31,PDE01,CPT,1.0,27.0,11A,0.0
1,2001-06-30,PDE01,CPT,1.0,28.0,11A,0.0
2,2001-09-30,PDE01,CPT,1.0,28.0,11A,0.0
3,2001-12-31,PDE01,CPT,1.0,28.0,11A,0.0
4,2002-03-31,PDE01,CPT,1.0,28.0,11A,0.0


In [4]:
# Explore the data structure
print("Unique ranks:")
print(df_clean['rank_pde'].value_counts())

print("\nUnique job codes:")
print(df_clean['job_code'].value_counts())

print("\nSex distribution:")
print(df_clean['sex'].value_counts())

print("\nMarriage status distribution:")
print(df_clean['married'].value_counts())


Unique ranks:
rank_pde
CPT    239
MAJ     60
Name: count, dtype: int64

Unique job codes:
job_code
11A    158
12A    107
35A     34
Name: count, dtype: int64

Sex distribution:
sex
1.0    219
0.0     80
Name: count, dtype: int64

Marriage status distribution:
married
0.0    151
1.0    148
Name: count, dtype: int64


In [5]:
# Study end date
STUDY_END = pd.to_datetime('2007-09-30')

# Analyze each officer's career trajectory
officer_summary = []

for officer in df_clean['pid_pde'].unique():
    officer_data = df_clean[df_clean['pid_pde'] == officer].copy()
    
    # Basic info
    first_date = officer_data['snpsht_dt'].min()
    last_date = officer_data['snpsht_dt'].max()
    
    # Check if promoted to Major
    promoted_to_major = (officer_data['rank_pde'] == 'MAJ').any()
    
    if promoted_to_major:
        # Find first Major promotion date
        major_date = officer_data[officer_data['rank_pde'] == 'MAJ']['snpsht_dt'].min()
        event = 1
        event_date = major_date
        censoring_type = 'promoted'
    else:
        # Not promoted - determine censoring type
        event = 0
        event_date = last_date
        if last_date < STUDY_END:
            censoring_type = 'left_early'
        else:
            censoring_type = 'study_end'
    
    # Calculate time to event (in days)
    time_to_event = (event_date - first_date).days
    
    # Get baseline characteristics
    baseline = officer_data.iloc[0]
    
    # Check for marriage during study
    marriage_changes = officer_data['married'].diff().fillna(0)
    got_married_during_study = (marriage_changes == 1).any()
    baseline_married = baseline['married']
    
    # Check for job code changes
    job_codes = officer_data['job_code'].unique()
    job_code_changed = len(job_codes) > 1
    baseline_job_code = baseline['job_code']
    
    officer_summary.append({
        'pid_pde': officer,
        'start_date': first_date,
        'event_date': event_date,
        'time_to_event_days': time_to_event,
        'event': event,
        'censoring_type': censoring_type,
        'sex': baseline['sex'],
        'baseline_age': baseline['age'],
        'baseline_married': baseline_married,
        'got_married_during_study': got_married_during_study,
        'baseline_job_code': baseline_job_code,
        'job_code_changed': job_code_changed,
        'all_job_codes': ', '.join(job_codes)
    })

# Convert to DataFrame
survival_df = pd.DataFrame(officer_summary)

print(f"Officer-level survival data shape: {survival_df.shape}")
print("\nEvent distribution:")
print(survival_df['censoring_type'].value_counts())

survival_df.head()


Officer-level survival data shape: (13, 13)

Event distribution:
censoring_type
promoted      7
left_early    4
study_end     2
Name: count, dtype: int64


,pid_pde,start_date,event_date,time_to_event_days,event,censoring_type,sex,baseline_age,baseline_married,got_married_during_study,baseline_job_code,job_code_changed,all_job_codes
0,PDE01,2001-03-31,2005-09-30,1644,1,promoted,1.0,27.0,0.0,True,11A,False,11A
1,PDE02,2001-03-31,2004-06-30,1187,1,promoted,1.0,27.0,0.0,False,11A,False,11A
2,PDE03,2001-03-31,2007-03-31,2191,1,promoted,1.0,33.0,1.0,False,11A,False,11A
3,PDE04,2001-03-31,2005-09-30,1644,1,promoted,0.0,27.0,0.0,True,12A,False,12A
4,PDE05,2001-03-31,2004-06-30,1187,1,promoted,0.0,24.0,0.0,True,12A,False,12A


In [6]:
# Summary statistics
print("=== SURVIVAL ANALYSIS SUMMARY ===")
print(f"Total officers: {len(survival_df)}")
print(f"Promoted to Major: {survival_df['event'].sum()} ({survival_df['event'].mean():.2%})")
print(f"Censored: {(1-survival_df['event']).sum()}")

print("\nCensoring breakdown:")
print(survival_df['censoring_type'].value_counts())

print("\nTime to event/censoring (days):")
print(survival_df['time_to_event_days'].describe())

print("\nBaseline characteristics:")
print(f"Male: {survival_df['sex'].sum()} ({survival_df['sex'].mean():.2%})")
print(f"Baseline married: {survival_df['baseline_married'].sum()} ({survival_df['baseline_married'].mean():.2%})")
print(f"Got married during study: {survival_df['got_married_during_study'].sum()} ({survival_df['got_married_during_study'].mean():.2%})")
print(f"Changed job code: {survival_df['job_code_changed'].sum()} ({survival_df['job_code_changed'].mean():.2%})")

print("\nBaseline age:")
print(survival_df['baseline_age'].describe())


=== SURVIVAL ANALYSIS SUMMARY ===
Total officers: 13
Promoted to Major: 7 (53.85%)
Censored: 6

Censoring breakdown:
censoring_type
promoted      7
left_early    4
study_end     2
Name: count, dtype: int64

Time to event/censoring (days):
count      13.000000
mean     1636.692308
std       516.991519
min      1096.000000
25%      1187.000000
50%      1552.000000
75%      2191.000000
max      2374.000000
Name: time_to_event_days, dtype: float64

Baseline characteristics:
Male: 9.0 (69.23%)
Baseline married: 2.0 (15.38%)
Got married during study: 7 (53.85%)
Changed job code: 3 (23.08%)

Baseline age:
count    13.000000
mean     27.000000
std       2.160247
min      24.000000
25%      27.000000
50%      27.000000
75%      27.000000
max      33.000000
Name: baseline_age, dtype: float64


In [7]:
# Create time-dependent covariate dataset for Cox model
# This requires expanding each officer's data for time-varying covariates

cox_data_list = []

for officer in df_clean['pid_pde'].unique():
    officer_data = df_clean[df_clean['pid_pde'] == officer].copy().reset_index(drop=True)
    
    # Determine event information
    promoted_to_major = (officer_data['rank_pde'] == 'MAJ').any()
    
    if promoted_to_major:
        # Find promotion event
        major_rows = officer_data[officer_data['rank_pde'] == 'MAJ']
        promotion_date = major_rows['snpsht_dt'].min()
        # Only include data up to promotion
        officer_data = officer_data[officer_data['snpsht_dt'] <= promotion_date]
    
    start_date = officer_data['snpsht_dt'].min()
    
    # Create time intervals
    for i in range(len(officer_data)):
        current_row = officer_data.iloc[i]
        
        # Time start (days from first observation)
        t_start = (current_row['snpsht_dt'] - start_date).days
        
        # Time stop
        if i < len(officer_data) - 1:
            next_row = officer_data.iloc[i + 1]
            t_stop = (next_row['snpsht_dt'] - start_date).days
            event_this_interval = 0  # No event in intermediate intervals
        else:
            # Last observation
            if promoted_to_major and current_row['rank_pde'] == 'MAJ':
                # Promotion event
                t_stop = (current_row['snpsht_dt'] - start_date).days
                event_this_interval = 1
            else:
                # Censored - extend to next quarter or study end
                if current_row['snpsht_dt'] < STUDY_END:
                    # Add 90 days for next quarter
                    t_stop = (current_row['snpsht_dt'] - start_date).days + 90
                else:
                    t_stop = (STUDY_END - start_date).days
                event_this_interval = 0
        
        # Skip intervals with zero duration
        if t_stop <= t_start:
            continue
            
        cox_data_list.append({
            'pid_pde': current_row['pid_pde'],
            't_start': t_start,
            't_stop': t_stop,
            'event': event_this_interval,
            'sex': current_row['sex'],
            'age': current_row['age'],
            'married': current_row['married'],
            'job_code': current_row['job_code']
        })

# Convert to DataFrame
cox_df = pd.DataFrame(cox_data_list)

print(f"Cox model dataset shape: {cox_df.shape}")
print(f"Total events: {cox_df['event'].sum()}")
print(f"Unique officers: {cox_df['pid_pde'].nunique()}")

cox_df.head(10)


Cox model dataset shape: (237, 8)
Total events: 0
Unique officers: 13


,pid_pde,t_start,t_stop,event,sex,age,married,job_code
0,PDE01,0,91,0,1.0,27.0,0.0,11A
1,PDE01,91,183,0,1.0,28.0,0.0,11A
2,PDE01,183,275,0,1.0,28.0,0.0,11A
3,PDE01,275,365,0,1.0,28.0,0.0,11A
4,PDE01,365,456,0,1.0,28.0,0.0,11A
5,PDE01,456,548,0,1.0,29.0,0.0,11A
6,PDE01,548,640,0,1.0,29.0,0.0,11A
7,PDE01,640,730,0,1.0,29.0,0.0,11A
8,PDE01,730,821,0,1.0,29.0,0.0,11A
9,PDE01,821,913,0,1.0,30.0,0.0,11A


In [8]:
# Add duration column for convenience
cox_df['duration'] = cox_df['t_stop'] - cox_df['t_start']

# Create dummy variables for job codes
job_dummies = pd.get_dummies(cox_df['job_code'], prefix='job')
cox_df = pd.concat([cox_df, job_dummies], axis=1)

print("Job code distribution:")
print(cox_df['job_code'].value_counts())

print("\nCox model data summary:")
print(cox_df.describe())


Job code distribution:
job_code
11A    131
12A     83
35A     23
Name: count, dtype: int64

Cox model data summary:
           t_start       t_stop  event         sex         age     married  \
count   237.000000   237.000000  237.0  237.000000  237.000000  237.000000   
mean    853.890295   945.185654    0.0    0.759494   30.025316    0.447257   
std     591.555101   591.526658    0.0    0.428295    2.987153    0.498263   
min       0.000000    91.000000    0.0    0.000000   24.000000    0.000000   
25%     365.000000   456.000000    0.0    1.000000   28.000000    0.000000   
50%     821.000000   913.000000    0.0    1.000000   30.000000    0.000000   
75%    1187.000000  1279.000000    0.0    1.000000   31.000000    1.000000   
max    2282.000000  2374.000000    0.0    1.000000   39.000000    1.000000   

         duration  
count  237.000000  
mean    91.295359  
std      0.768349  
min     90.000000  
25%     91.000000  
50%     91.000000  
75%     92.000000  
max     92.000000  


In [9]:
# Fit Cox Proportional Hazards Model
cph = CoxPHFitter()

# Select covariates for the model
# Using sex, age, married status, and main job codes
job_cols = [col for col in cox_df.columns if col.startswith('job_')]
covariates = ['sex', 'age', 'married'] + job_cols

# Prepare data for model fitting
model_data = cox_df[['t_start', 't_stop', 'event'] + covariates].copy()

print("Fitting Cox model with covariates:")
print(covariates)

# Fit the model
cph.fit(model_data, duration_col='t_stop', event_col='event', 
        start_col='t_start', show_progress=True)

# Display results
print("\n=== COX MODEL RESULTS ===")
cph.print_summary()


Fitting Cox model with covariates:
['sex', 'age', 'married', 'job_code', 'job_11A', 'job_12A', 'job_35A']


TypeError: CoxPHFitter.fit() got an unexpected keyword argument 'start_col'

In [ ]:
# Plot hazard ratios
plt.figure(figsize=(10, 8))
cph.plot()
plt.title('Hazard Ratios for Promotion to Major')
plt.xlabel('Hazard Ratio (log scale)')
plt.tight_layout()
plt.show()

# Get hazard ratios and confidence intervals
hazard_ratios = np.exp(cph.params_)
conf_int = np.exp(cph.confidence_intervals_)

print("\n=== HAZARD RATIOS ===")
results_df = pd.DataFrame({
    'Hazard_Ratio': hazard_ratios,
    'Lower_CI': conf_int.iloc[:, 0],
    'Upper_CI': conf_int.iloc[:, 1],
    'P_Value': cph.summary['p']
})

results_df['Significant'] = results_df['P_Value'] < 0.05
print(results_df.round(4))


In [ ]:
# Model diagnostics
print("=== MODEL DIAGNOSTICS ===")
print(f"Concordance Index: {cph.concordance_index_:.4f}")
print(f"Log-likelihood: {cph.log_likelihood_:.4f}")
print(f"AIC: {cph.AIC_:.4f}")

# Check proportional hazards assumption
try:
    cph.check_assumptions(model_data, show_plots=True)
except Exception as e:
    print(f"Could not run proportional hazards test: {e}")


In [ ]:
# Kaplan-Meier curves for key variables
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Convert back to simple survival format for KM curves
km_data = survival_df.copy()
km_data['duration_years'] = km_data['time_to_event_days'] / 365.25

# Sex
kmf = KaplanMeierFitter()
for sex, label in [(0, 'Female'), (1, 'Male')]:
    mask = km_data['sex'] == sex
    kmf.fit(km_data.loc[mask, 'duration_years'], 
            km_data.loc[mask, 'event'], label=label)
    kmf.plot_survival_function(ax=axes[0,0])
axes[0,0].set_title('Survival by Sex')
axes[0,0].set_ylabel('Probability of Remaining Captain')

# Baseline marriage status
for married, label in [(0, 'Unmarried at baseline'), (1, 'Married at baseline')]:
    mask = km_data['baseline_married'] == married
    kmf.fit(km_data.loc[mask, 'duration_years'], 
            km_data.loc[mask, 'event'], label=label)
    kmf.plot_survival_function(ax=axes[0,1])
axes[0,1].set_title('Survival by Baseline Marriage Status')
axes[0,1].set_ylabel('Probability of Remaining Captain')

# Job code change
for changed, label in [(False, 'No job change'), (True, 'Job code changed')]:
    mask = km_data['job_code_changed'] == changed
    kmf.fit(km_data.loc[mask, 'duration_years'], 
            km_data.loc[mask, 'event'], label=label)
    kmf.plot_survival_function(ax=axes[1,0])
axes[1,0].set_title('Survival by Job Code Change')
axes[1,0].set_ylabel('Probability of Remaining Captain')

# Age groups
km_data['age_group'] = pd.cut(km_data['baseline_age'], 
                              bins=[0, 27, 30, 100], 
                              labels=['≤27', '28-30', '>30'])
for age_group in km_data['age_group'].cat.categories:
    mask = km_data['age_group'] == age_group
    kmf.fit(km_data.loc[mask, 'duration_years'], 
            km_data.loc[mask, 'event'], label=f'Age {age_group}')
    kmf.plot_survival_function(ax=axes[1,1])
axes[1,1].set_title('Survival by Baseline Age Group')
axes[1,1].set_ylabel('Probability of Remaining Captain')

for ax in axes.flat:
    ax.set_xlabel('Years as Captain')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Interpretation of results
print("=== MODEL INTERPRETATION ===")
print("\nKey findings:")

significant_vars = results_df[results_df['Significant']].index
if len(significant_vars) > 0:
    print("\nStatistically significant factors (p < 0.05):")
    for var in significant_vars:
        hr = results_df.loc[var, 'Hazard_Ratio']
        p_val = results_df.loc[var, 'P_Value']
        if hr > 1:
            effect = f"increases promotion hazard by {(hr-1)*100:.1f}%"
        else:
            effect = f"decreases promotion hazard by {(1-hr)*100:.1f}%"
        print(f"  - {var}: HR = {hr:.3f}, p = {p_val:.4f} - {effect}")
else:
    print("No statistically significant factors found at p < 0.05 level.")

print("\nNote: Hazard ratio > 1 means increased risk of promotion (shorter time to Major)")
print("      Hazard ratio < 1 means decreased risk of promotion (longer time to Major)")
